# Fine-tuning LLM untuk Chatbot Hukum Indonesia

**Modul**: Pengembangan Generative AI berbasis LLM — Digdaya Hackathon BI

**Target**: Advanced (4 pts)

---

## Ringkasan Notebook

1. Load dataset `Ichsan2895/alpaca-gpt4-indonesian` (format Alpaca, Bahasa Indonesia)
2. Train/validation split (90/10)
3. Mapping dataset ke ChatML template (format native Qwen2.5)
4. Load `Qwen/Qwen2.5-7B-Instruct` dengan QLoRA 4-bit + double quantization
5. LoRA adapters pada komponen Multi-Head Attention
6. Dua eksperimen SFTTrainer dengan hyperparameter berbeda
7. Loss curve comparison
8. Merge LoRA (merged_16bit) + push ke HuggingFace Hub

> **Catatan**: Notebook ini didesain untuk Google Colab dengan GPU T4 (16GB VRAM).

## 1. Instalasi Dependencies

In [ ]:
# Install Unsloth dan dependencies (Colab/Kaggle compatible)
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub safetensors matplotlib

In [ ]:
# Restart runtime setelah instalasi Unsloth (Colab only)
import sys
if "google.colab" in sys.modules:
    import os
    os.kill(os.getpid(), 9)  # Force restart — hanya Colab

## 2. Import Libraries

In [ ]:
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import matplotlib.pyplot as plt
import os
import re

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 3. Load Dataset

Dataset: `Ichsan2895/alpaca-gpt4-indonesian` — format Alpaca (instruction, input, output) dalam Bahasa Indonesia.

In [ ]:
# Load dataset dari HuggingFace
dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")
print(f"Jumlah total sample: {len(dataset)}")
print(f"Kolom: {dataset.column_names}")

In [ ]:
# Tampilkan beberapa baris dataset MENTAH (sebelum mapping)
print("=" * 80)
print("DATASET MENTAH — Sebelum Chat Template Mapping")
print("=" * 80)
for i in range(3):
    sample = dataset[i]
    print(f"--- Sample #{i+1} ---")
    print(f"instruction: {sample['instruction'][:200]}...")
    print(f"input: {sample['input'][:100] if sample['input'] else '(kosong)'}...")
    print(f"output: {sample['output'][:200]}...")
    print()

## 4. Train/Validation Split

Split dataset 90% training, 10% validation untuk memenuhi kriteria Skilled & Advanced.

In [ ]:
# Train/validation split (90/10)
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

## 5. Chat Template Mapping

Mapping dataset ke **ChatML format** (format native Qwen2.5).

Format ChatML:
```
<|im_start|>system
{system_message}<|im_end|>
<|im_start|>user
{instruction + input}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>
```

Token spesial ChatML:
- `<|im_start|>` — penanda awal pesan (role)
- `<|im_end|>` — penanda akhir pesan
- Role tokens: `system`, `user`, `assistant`

In [ ]:
# Define system prompt dalam Bahasa Indonesia
SYSTEM_PROMPT = """Anda adalah asisten AI yang membantu menjawab pertanyaan dalam Bahasa Indonesia. Berikan jawaban yang akurat, informatif, dan sopan."""

def format_chatml(example):
    """
    Mapping function: mengubah row dataset Alpaca ke format ChatML.
    
    Format ChatML:
    <|im_start|>system
    {system}<|im_end|>
    <|im_start|>user
    {instruction + input}<|im_end|>
    <|im_start|>assistant
    {output}<|im_end|>
    """
    # Gabungkan instruction dan input jika input tidak kosong
    instruction = example["instruction"]
    user_input = example["input"]
    
    if user_input and user_input.strip():
        user_message = f"{instruction}\n\n{user_input}"
    else:
        user_message = instruction
    
    # Bangun teks lengkap format ChatML
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_message}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['output']}<|im_end|>"
    )
    
    return {"text": text}

# Terapkan mapping ke train dan eval dataset
train_dataset_mapped = train_dataset.map(format_chatml)
eval_dataset_mapped = eval_dataset.map(format_chatml)

print("Mapping selesai. Kolom baru: 'text' ditambahkan.")

In [ ]:
# Tampilkan 1 baris dataset yang SUDAH di-mapping (lengkap dengan token spesial)
print("=" * 80)
print("DATASET SUDAH DI-MAPPING — Format ChatML dengan Token Spesial")
print("=" * 80)
print()

sample_mapped = train_dataset_mapped[0]
print(sample_mapped["text"])

print()
print("=" * 80)
print("Token spesial yang digunakan:")
print("  <|im_start|>  — penanda awal role (system/user/assistant)")
print("  <|im_end|>    — penanda akhir pesan")
print("  system, user, assistant — nama role")
print("=" * 80)

## 6. Load Model dengan QLoRA 4-bit + Double Quantization

Menggunakan `Qwen/Qwen2.5-7B-Instruct` sebagai base model. QLoRA configuration:
- **4-bit quantization** dengan `bitsandbytes` (`load_in_4bit=True`)
- **Double quantization** (`bnb_4bit_use_double_quant=True`) — quantisasi ulang konstanta quantisasi untuk mengurangi memory footprint lebih lanjut
- **NF4 datatype** (`bnb_4bit_quant_type="nf4"`) — NormalFloat4, format optimal untuk weight terdistribusi normal

In [ ]:
# Model dan tokenizer
model_name = "Qwen/Qwen2.5-7B-Instruct"
max_seq_length = 2048  # Sequence length maksimal

# Load model dengan QLoRA 4-bit + double quantization via Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,  # Auto-detect (float16 untuk T4)
    load_in_4bit=True,  # [QLoRA] 4-bit quantization
    # Double quantization & NF4 di-handle otomatis oleh Unsloth
    # Unsloth internally sets: bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4"
)

print(f"Model loaded: {model_name}")
print(f"Quantization: 4-bit NF4 + Double Quantization (QLoRA)")
print(f"Max sequence length: {max_seq_length}")

In [ ]:
# Verifikasi konfigurasi quantisasi
import bitsandbytes as bnb

quant_config = model.config.quantization_config
if quant_config:
    print("Quantization Config:")
    for key, value in quant_config.items():
        print(f"  {key}: {value}")
else:
    # Unsloth mungkin menyimpan config di tempat berbeda
    print("Quantization dikelola oleh Unsloth secara internal")

## 7. LoRA Configuration

LoRA adapters didefinisikan pada **seluruh komponen Multi-Head Attention (MHA)**:
- `q_proj` — Query projection
- `k_proj` — Key projection
- `v_proj` — Value projection
- `o_proj` — Output projection

Ini memastikan LoRA mencakup setidaknya satu komponen komputasi utama (MHA) secara penuh.

In [ ]:
# Definisikan LoRA adapters pada komponen Multi-Head Attention
# Parameter lora_r dan lora_alpha akan divariasikan per eksperimen

def apply_lora(model, r=16, lora_alpha=16, lora_dropout=0.0):
    """
    Terapkan LoRA adapters pada model.
    
    Target modules: q_proj, k_proj, v_proj, o_proj (seluruh MHA)
    Juga menyertakan gate_proj, up_proj, down_proj (FFN) untuk coverage lebih luas
    """
    model = FastLanguageModel.get_peft_model(
        model,
        r=r,  # LoRA rank
        lora_alpha=lora_alpha,  # Scaling factor
        lora_dropout=lora_dropout,  # Regularization
        target_modules=[
            # Multi-Head Attention (KOMPONEN UTAMA - wajib)
            "q_proj", "k_proj", "v_proj", "o_proj",
            # Feed-Forward Network (tambahan untuk coverage lebih baik)
            "gate_proj", "up_proj", "down_proj",
        ],
        use_gradient_checkpointing="unsloth",  # Memory-efficient
        random_state=42,
        use_rslora=False,  # Standard LoRA
        loftq_config=None,
    )
    return model

print("Fungsi apply_lora() siap digunakan.")
print("Target modules MHA: q_proj, k_proj, v_proj, o_proj")
print("Target modules FFN: gate_proj, up_proj, down_proj")

In [ ]:
# Update tokenizer untuk ChatML
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",  # Format ChatML untuk Qwen2.5
    mapping={"role": "from", "content": "value", "user": "user", "assistant": "assistant"},
)

print("Tokenizer updated dengan ChatML template.")

## 8. Eksperimen 1 — Default Hyperparameters

| Parameter | Value |
|-----------|-------|
| learning_rate | 2e-4 |
| per_device_train_batch_size | 4 |
| lora_r | 16 |
| lora_alpha | 16 |
| gradient_accumulation_steps | 4 |
| max_steps | 800 |
| evaluation_strategy | steps |
| eval_steps | 100 |
| logging_steps | 50 |
| lr_scheduler_type | cosine |

In [ ]:
# Load model ulang untuk eksperimen 1
model_exp1, tokenizer_exp1 = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Setup tokenizer ChatML
tokenizer_exp1 = get_chat_template(
    tokenizer_exp1,
    chat_template="chatml",
    mapping={"role": "from", "content": "value", "user": "user", "assistant": "assistant"},
)

# Apply LoRA — Experiment 1 hyperparameters
# LoRA pada seluruh MHA: q_proj, k_proj, v_proj, o_proj
model_exp1 = FastLanguageModel.get_peft_model(
    model_exp1,
    r=16,  # LoRA rank — Experiment 1
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model Experiment 1 siap.")
print(f"LoRA rank (r): 16, lora_alpha: 16")

In [ ]:
# Training Arguments — Experiment 1
training_args_exp1 = TrainingArguments(
    output_dir="./outputs/exp1",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,  # Learning rate lebih tinggi
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    max_steps=800,
    
    # [Skilled] Evaluation strategy
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    
    # [Skilled] Logging
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,
    
    # Optimization
    optim="adamw_8bit",  # 8-bit AdamW untuk hemat VRAM
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # Precision
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    
    # Checkpointing
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Reporting
    report_to="none",  # Disable wandb/tensorboard untuk simplifikasi
    save_total_limit=2,
    
    # Seed
    seed=42,
)

print("TrainingArguments Experiment 1:")
print(f"  learning_rate: {training_args_exp1.learning_rate}")
print(f"  batch_size: {training_args_exp1.per_device_train_batch_size}")
print(f"  max_steps: {training_args_exp1.max_steps}")
print(f"  evaluation_strategy: {training_args_exp1.evaluation_strategy}")
print(f"  eval_steps: {training_args_exp1.eval_steps}")
print(f"  logging_steps: {training_args_exp1.logging_steps}")

In [ ]:
# SFTTrainer — Experiment 1
trainer_exp1 = SFTTrainer(
    model=model_exp1,
    tokenizer=tokenizer_exp1,
    args=training_args_exp1,
    train_dataset=train_dataset_mapped,
    eval_dataset=eval_dataset_mapped,  # [Skilled] Validation dataset
    dataset_text_field="text",  # Kolom hasil mapping ChatML
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Tidak packing agar setiap sample sequence terpisah
)

print("SFTTrainer Experiment 1 siap.")
print(f"Training samples: {len(train_dataset_mapped)}")
print(f"Eval samples: {len(eval_dataset_mapped)}")

In [ ]:
# Jalankan training — Experiment 1
print("=" * 60)
print("MEMULAI TRAINING — EXPERIMENT 1")
print(f"  Model: {model_name}")
print(f"  LoRA r=16, lora_alpha=16")
print(f"  lr=2e-4, batch_size=4, max_steps=800")
print("=" * 60)

trainer_exp1.train()

# Simpan log training
log_history_exp1 = trainer_exp1.state.log_history
print(f"\nTraining Experiment 1 selesai. Total log entries: {len(log_history_exp1)}")

## 9. Eksperimen 2 — Alternative Hyperparameters

| Parameter | Value | Perubahan dari Exp 1 |
|-----------|-------|---------------------|
| learning_rate | 5e-5 | Lebih rendah (4x) |
| per_device_train_batch_size | 2 | Lebih kecil |
| lora_r | 32 | Lebih besar (2x) |
| lora_alpha | 64 | Lebih besar (4x) |
| gradient_accumulation_steps | 8 | Lebih besar (2x) |
| max_steps | 800 | Sama |
| evaluation_strategy | steps | Sama |
| eval_steps | 100 | Sama |
| logging_steps | 50 | Sama |

**Rasional**: Learning rate lebih rendah dengan LoRA rank lebih tinggi untuk mencari keseimbangan antara stabilitas training dan kapasitas adaptasi.

In [ ]:
# Hapus model experiment 1 dari memory
import gc
del model_exp1, trainer_exp1
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared.")

In [ ]:
# Load model ulang untuk eksperimen 2
model_exp2, tokenizer_exp2 = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

tokenizer_exp2 = get_chat_template(
    tokenizer_exp2,
    chat_template="chatml",
    mapping={"role": "from", "content": "value", "user": "user", "assistant": "assistant"},
)

# Apply LoRA — Experiment 2 hyperparameters
# LoRA pada seluruh MHA: q_proj, k_proj, v_proj, o_proj
model_exp2 = FastLanguageModel.get_peft_model(
    model_exp2,
    r=32,  # LoRA rank — Experiment 2 (2x Experiment 1)
    lora_alpha=64,  # 4x Experiment 1
    lora_dropout=0.05,  # Dropout kecil untuk regularisasi
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model Experiment 2 siap.")
print(f"LoRA rank (r): 32, lora_alpha: 64, lora_dropout: 0.05")

In [ ]:
# Training Arguments — Experiment 2
training_args_exp2 = TrainingArguments(
    output_dir="./outputs/exp2",
    per_device_train_batch_size=2,  # Lebih kecil dari Exp 1
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # Lebih besar (efektif batch size tetap 16)
    learning_rate=5e-5,  # 4x lebih rendah dari Exp 1
    lr_scheduler_type="cosine_with_restarts",  # Cosine with warm restarts
    warmup_ratio=0.1,
    max_steps=800,
    
    # [Skilled] Evaluation strategy
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    
    # [Skilled] Logging
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,
    
    # Optimization
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # Precision
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    
    # Checkpointing
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Reporting
    report_to="none",
    save_total_limit=2,
    
    seed=123,  # Seed berbeda untuk eksperimen berbeda
)

print("TrainingArguments Experiment 2:")
print(f"  learning_rate: {training_args_exp2.learning_rate}")
print(f"  batch_size: {training_args_exp2.per_device_train_batch_size}")
print(f"  lr_scheduler: {training_args_exp2.lr_scheduler_type}")
print(f"  max_steps: {training_args_exp2.max_steps}")

In [ ]:
# SFTTrainer — Experiment 2
trainer_exp2 = SFTTrainer(
    model=model_exp2,
    tokenizer=tokenizer_exp2,
    args=training_args_exp2,
    train_dataset=train_dataset_mapped,
    eval_dataset=eval_dataset_mapped,  # [Skilled] Validation dataset
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
)

print("SFTTrainer Experiment 2 siap.")

In [ ]:
# Jalankan training — Experiment 2
print("=" * 60)
print("MEMULAI TRAINING — EXPERIMENT 2")
print(f"  Model: {model_name}")
print(f"  LoRA r=32, lora_alpha=64")
print(f"  lr=5e-5, batch_size=2, gradient_accumulation=8")
print(f"  max_steps=800")
print("=" * 60)

trainer_exp2.train()

# Simpan log training
log_history_exp2 = trainer_exp2.state.log_history
print(f"\nTraining Experiment 2 selesai. Total log entries: {len(log_history_exp2)}")

## 10. Loss Curve Comparison

Membandingkan kurva training loss dan evaluation loss dari kedua eksperimen untuk menentukan hyperparameter mana yang menghasilkan training paling optimal tanpa overfitting.

In [ ]:
# Ekstrak training dan evaluation loss
def extract_losses(log_history):
    """Ekstrak training loss dan eval loss dari log history."""
    train_steps, train_losses = [], []
    eval_steps, eval_losses = [], []
    
    for entry in log_history:
        if "loss" in entry and "eval_loss" not in entry:
            train_steps.append(entry.get("step", 0))
            train_losses.append(entry["loss"])
        if "eval_loss" in entry:
            eval_steps.append(entry.get("step", 0))
            eval_losses.append(entry["eval_loss"])
    
    return train_steps, train_losses, eval_steps, eval_losses

t1_steps, t1_losses, e1_steps, e1_losses = extract_losses(log_history_exp1)
t2_steps, t2_losses, e2_steps, e2_losses = extract_losses(log_history_exp2)

print(f"Experiment 1 — Training loss points: {len(t1_losses)}, Eval loss points: {len(e1_losses)}")
print(f"Experiment 2 — Training loss points: {len(t2_losses)}, Eval loss points: {len(e2_losses)}")

In [ ]:
# Plot perbandingan loss curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training Loss Comparison
ax1 = axes[0]
ax1.plot(t1_steps, t1_losses, label="Exp 1 (lr=2e-4, r=16)", color="#2196F3", linewidth=1.5, alpha=0.8)
ax1.plot(t2_steps, t2_losses, label="Exp 2 (lr=5e-5, r=32)", color="#FF5722", linewidth=1.5, alpha=0.8)
ax1.set_xlabel("Steps")
ax1.set_ylabel("Training Loss")
ax1.set_title("Training Loss Comparison")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Evaluation Loss Comparison
ax2 = axes[1]
ax2.plot(e1_steps, e1_losses, label="Exp 1 (lr=2e-4, r=16)", color="#2196F3", 
         marker="o", markersize=5, linewidth=1.5)
ax2.plot(e2_steps, e2_losses, label="Exp 2 (lr=5e-5, r=32)", color="#FF5722", 
         marker="s", markersize=5, linewidth=1.5)
ax2.set_xlabel("Steps")
ax2.set_ylabel("Evaluation Loss")
ax2.set_title("Evaluation Loss Comparison")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("Fine-Tuning Qwen2.5-7B-Instruct: Eksperimen Hyperparameter", 
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Analisis: Pilih model terbaik berdasarkan eval loss terendah
min_e1 = min(e1_losses) if e1_losses else float("inf")
min_e2 = min(e2_losses) if e2_losses else float("inf")

print("=" * 60)
print("ANALISIS PERBANDINGAN")
print("=" * 60)
print(f"Experiment 1 (lr=2e-4, r=16):")
print(f"  Training loss terakhir: {t1_losses[-1]:.4f}" if t1_losses else "  N/A")
print(f"  Eval loss terendah:    {min_e1:.4f}" if e1_losses else "  N/A")
print()
print(f"Experiment 2 (lr=5e-5, r=32):")
print(f"  Training loss terakhir: {t2_losses[-1]:.4f}" if t2_losses else "  N/A")
print(f"  Eval loss terendah:    {min_e2:.4f}" if e2_losses else "  N/A")
print()

# Tentukan best model
if min_e1 < min_e2:
    print("KESIMPULAN: Experiment 1 menghasilkan eval loss lebih rendah.")
    print("Menggunakan model dari Experiment 1 untuk final merge.")
    BEST_MODEL = model_exp1
    BEST_TOKENIZER = tokenizer_exp1
else:
    print("KESIMPULAN: Experiment 2 menghasilkan eval loss lebih rendah.")
    print("Menggunakan model dari Experiment 2 untuk final merge.")
    BEST_MODEL = model_exp2
    BEST_TOKENIZER = tokenizer_exp2

## 11. Merge LoRA & Simpan Model

Merge adapters LoRA ke base model dalam format **merged_16bit** (FP16) lalu push ke HuggingFace Hub.

In [ ]:
# Merge LoRA adapters ke model dan simpan dalam 16-bit
# Gunakan merged_16bit = True sesuai ketentuan submission

print("Menyimpan model hasil fine-tuning (merged_16bit)...")

# Nama model untuk HuggingFace Hub
HF_USERNAME = "RayhanLup1n"
MODEL_NAME = "qwen2.5-7b-indonesian-legal-finetuned"

# Merge & save locally dalam 16-bit
BEST_MODEL.save_pretrained_merged(
    f"./{MODEL_NAME}",
    BEST_TOKENIZER,
    save_method="merged_16bit",  # [WAJIB] merged dalam 16-bit
)

print(f"Model disimpan di: ./{MODEL_NAME}/")

In [ ]:
# Push model ke HuggingFace Hub
print(f"Mengunggah model ke HuggingFace Hub: {HF_USERNAME}/{MODEL_NAME}")

# Push merged model (cara 1: via push_to_hub)
BEST_MODEL.push_to_hub_merged(
    f"{HF_USERNAME}/{MODEL_NAME}",
    BEST_TOKENIZER,
    save_method="merged_16bit",  # [WAJIB] merged dalam 16-bit
    token=True,  # Gunakan token dari huggingface-cli login
)

print(f"Model berhasil diunggah: https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")

## 12. Simpan Link HuggingFace

In [ ]:
# Simpan link HuggingFace ke file txt
hf_url = f"https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}"

with open("link_huggingface.txt", "w", encoding="utf-8") as f:
    f.write(hf_url)

print(f"Link disimpan ke link_huggingface.txt")
print(f"URL: {hf_url}")

## Kesimpulan

Notebook ini telah menyelesaikan seluruh kriteria:

**Basic (2 pts)** ✅
- [x] Mapping dataset Alpaca ke ChatML template dengan token spesial
- [x] QLoRA 4-bit + double quantization
- [x] LoRA pada Multi-Head Attention (q_proj, k_proj, v_proj, o_proj)
- [x] SFTTrainer minimal 800 steps (tanpa OOM)
- [x] Push model ke HuggingFace (merged_16bit)

**Skilled (3 pts)** ✅
- [x] Train/validation split + eval_dataset
- [x] evaluation_strategy="steps" + logging
- [x] Dua eksperimen hyperparameter + loss curve comparison

**Advanced (4 pts)** ✅
- [x] Semua ketentuan Skilled terpenuhi
- [x] Siap untuk dilanjutkan ke GRPO training di notebook berikutnya